# 10. Personal de obra

La empresa PQR ha obtenido un contrato para pavimentar las calles de una parte del centro de la ciudad de Rivertown. El Departamento de Ingenierı́a de Tráfico y Planificación Urbana estimó que **se requiere ==🔵al menos== el número de empleados indicado en la Tabla 9** en cada **intervalo de cuatro horas de un dı́a estándar de 24 horas.**

| Periodo | Personal requerido |
| :-----: | :----------------: |
|  06–10  |         70         |
|  10–14  |         20         |
|  14–18  |         80         |
|  18–22  |         30         |
|  22–02  |         10         |
|  02–06  |         10         |

*Tabla 9: Personal requerido en cada intervalo de cuatro horas.*

Todos los empleados de PQR trabajan en **turnos continuos de ocho horas**. Existen **seis turnos posibles**, cada uno de los cuales comienza al inicio de alguno de los intervalos de cuatro horas señalados en la tabla. Todos los empleados reciben el mismo salario por hora, salvo quienes trabajan en los **turnos que inician a las 20:00 h y a las 00:00 h, que reciben un aumento del 50 %.** Además, **el salario por hora se duplica entre las 00:00 h y las 06:00 h.**

**a)** Construya un modelo de programación lineal entera que determine **cuántos empleados se requieren en cada uno de los seis turnos** para **minimizar el costo salarial** de PQR y cubrir los requerimientos de personal.

**b)** La empresa de trabajo temporal TrabajoFlex ofrece horarios más flexibles, con turnos de cuatro horas que comienzan al inicio de cada uno de los intervalos indicados en la tabla. El costo de emplear personal de TrabajoFlex es el mismo en todos los horarios, pero resulta un 80 % más alto que el costo diario del personal de PQR en cada turno. PQR desea saber si le convendrı́a utilizar a TrabajoFlex. Modifique el modelo anterior para encontrar la combinación óptima de personal de PQR y de TrabajoFlex que minimice el costo salarial y cumpla con los requerimientos.


---
### preguntas que me surgieron
*  los **turnos que inician a las 20:00 h y a las 00:00 h ???** ➡️ no existen turnos que inicien a las 20:00 ni a las 00:00. (recordemos: *"Todos los empleados de PQR trabajan en **turnos continuos de ocho horas**. **Existen seis turnos** posibles"*)
* "TrabajoFlex ofrece horarios más flexibles, con turnos de cuatro horas que comienzan al inicio de cada uno de los intervalos indicados en la tabla" ⬅️ no es lo mismo??? ➡️ no, TrabajoFlex ofrece serivios por medios turnos = 1 periodo de 4 horas.

---


## Datos y supuestos  
* turnos continuos de ocho horas
* 6 turnos posibles, cada uno de los cuales comienza al inicio de alguno de los intervalos de cuatro horas señalados en la tabla
*  turnos que inician a las 20:00 h y a las 00:00 h, que reciben un aumento del 50 %. -> no pasa
* el salario por hora se duplica entre las 00:00 h y las 06:00 h.
*  cuántos empleados se requieren en cada uno de los seis turnos para minimizar el costo salarial

----

In [ ]:
import numpy as np

from docplex.mp.model import Model

mdl = Model(name='Personal de obra')

# Para el a)
## Formulación matemática 
### Variables de decisión

Numero de trabajadores que inician su jornada/turno en el periodo $i \ \epsilon\  \{1, 2, 3, 4, 5, 6\}$
$$x_i$$


In [ ]:
turnos_i = [1, 2, 3, 4, 5, 6]
turnos_i = np.linspace(1, 6, 6, dtype=int)

In [18]:
x = mdl.integer_var_list(turnos_i, lb=0, name='x')

In [19]:
x

[docplex.mp.Var(type=I,name='x_1'),
 docplex.mp.Var(type=I,name='x_2'),
 docplex.mp.Var(type=I,name='x_3'),
 docplex.mp.Var(type=I,name='x_4'),
 docplex.mp.Var(type=I,name='x_5'),
 docplex.mp.Var(type=I,name='x_6')]

In [24]:
x[0]

docplex.mp.Var(type=I,name='x_1')

### Restricciones

####  Demanda 
De acuerdo a la tabla

| Periodo | Personal requerido | i=  | lo cubren    |
| :-----: | :----------------: | --- | ------------ |
|  06–10  |         70         | 1   | $x_6$, $x_1$ |
|  10–14  |         20         | 2   | $x_1$, $x_2$ |
|  14–18  |         80         | 3   | $x_2$, $x_3$ |
|  18–22  |         30         | 4   | $x_3$, $x_4$ |
|  22–02  |         10         | 5   | $x_4$, $x_5$ |
|  02–06  |         10         | 6   | $x_5$, $x_6$ |

$$ x_6 + x_1 \ge 70$$
$$ x_1 + x_2 \ge 20$$
$$ x_2 + x_3 \ge 80$$
$$ x_3 + x_4 \ge 30$$
$$ x_4 + x_5 \ge 10$$
$$ x_5 + x_6 \ge 10$$

In [27]:
mdl.add_constraints([
    x[5] + x[0] >= 70,  # Turno 6 + Turno 1
    x[0] + x[1] >= 20,  # Turno 1 + Turno 2
    x[1] + x[2] >= 80,  # Turno 2 + Turno 3
    x[2] + x[3] >= 30,  # Turno 3 + Turno 4
    x[3] + x[4] >= 10,  # Turno 4 + Turno 5
    x[4] + x[5] >= 10   # Turno 5 + Turno 6
])

[docplex.mp.LinearConstraint[](x_1+x_6,GE,70),
 docplex.mp.LinearConstraint[](x_1+x_2,GE,20),
 docplex.mp.LinearConstraint[](x_2+x_3,GE,80),
 docplex.mp.LinearConstraint[](x_3+x_4,GE,30),
 docplex.mp.LinearConstraint[](x_4+x_5,GE,10),
 docplex.mp.LinearConstraint[](x_5+x_6,GE,10)]

## Función objetivo 
Cuántos empleados se requieren en cada uno de los seis turnos para minimizar el costo salarial de PQR
* Sea el salario por hora regular S=1
* el salario por hora se duplica entre las 00:00 h y las 06:00 h.

$$\min z= 8x_1 + 8x_2+8x_3 + (6 + 2\times2)x_4 + (2 + 2\times6)x_5 + (2\times4 +4 )x_6​$$

es decir 

$$\min z= 8(x_1 + x_2 + x_3) + 10x_4 + 14x_5 + 12x_6​$$

In [28]:
# Funcion objetivo
mdl.minimize(
    mdl.sum( 8*(x[0] + x[1] + x[2])+ 10*x[3] + 14*x[4] + 12*x[5] )
)


# Resolver e imprimir solucion
solution = mdl.solve(log_output=True)

print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 1680.000000 after 0.01 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 6 rows, 6 columns, and 12 nonzeros.
Reduced MIP has 0 binaries, 6 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.01 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 6 rows, 6 columns, and 12 nonzeros.
Reduced MIP has 0 binaries, 6 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.01 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 4 threads.
Root relaxation solution time = 0.01 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                         1680.0000        0.0000           100.00%
*     0+    0              